# Galileo's Letters on Sunspots (1613) — Implementation
# 갈릴레오의 태양 흑점 편지 (1613) — 구현

This notebook visualizes the key geometric arguments Galileo used to prove
that sunspots are features on the Sun's surface, not orbiting satellites.

이 노트북은 Galileo가 흑점이 태양 표면의 현상임을 증명하기 위해 사용한
핵심 기하학적 논증을 시각화합니다.

**Contents / 목차:**
1. Solar Projection Setup — 태양 투영 기본 설정
2. Foreshortening Effect — 원근 단축 효과
3. Apparent Motion on the Solar Disk — 태양 원반 위의 겉보기 운동
4. Scheiner's Satellite Hypothesis vs. Galileo's Surface Model — Scheiner 위성 가설 vs. Galileo 표면 모델
5. Simulating Sunspot Observations — 흑점 관측 시뮬레이션
6. Solar Rotation Period Estimation — 태양 자전 주기 추정
7. Connection to Modern Solar Physics — 현대 태양 물리학과의 연결

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({
    "figure.figsize": (10, 8),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

## Part 1: Foreshortening Effect / 원근 단축 효과

Galileo's strongest argument: sunspots appear **compressed** near the limb.
A circular spot at the Sun's center looks like a thin ellipse near the edge.

$$\text{Apparent width} = w_0 \cos\theta$$

where $w_0$ is the true width, $\theta$ is the angle from disk center.

This only happens if spots are **on** the spherical surface — orbiting satellites
would maintain their circular shape at all positions.

Galileo의 가장 강력한 논증: 흑점이 가장자리(limb) 근처에서 **찌그러져** 보입니다.
태양 중심에서 원형인 흑점이 가장자리에서는 얇은 타원으로 보입니다.
이는 흑점이 구면 **위에** 있을 때만 발생하는 현상입니다.

In [ ]:
def draw_solar_disk_with_foreshortening():
    """Demonstrate foreshortening of sunspots on the solar disk."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # --- Left panel: Solar disk with foreshortened spots ---
    ax = axes[0]
    sun = plt.Circle((0, 0), 1.0, color="#FDB813", ec="darkorange", lw=2)
    ax.add_patch(sun)

    # Place spots at different longitudes (theta from center)
    spot_angles = np.array([0, 20, 40, 60, 75, 85])  # degrees
    spot_radius = 0.06  # true radius on surface

    for theta_deg in spot_angles:
        theta = np.radians(theta_deg)
        x = np.sin(theta)
        y = 0.0

        # Foreshortened width (east-west compressed), height unchanged
        w = spot_radius * np.cos(theta)
        h = spot_radius

        ellipse = patches.Ellipse(
            (x, y), width=2 * w, height=2 * h,
            color="black", alpha=0.8
        )
        ax.add_patch(ellipse)
        ax.annotate(
            f"{theta_deg}°", (x, y - 0.12),
            ha="center", fontsize=9, color="white",
            bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.6),
        )

    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_aspect("equal")
    ax.set_title("Solar Disk: Foreshortening of Sunspots\n태양 원반: 흑점의 원근 단축")
    ax.set_xlabel("East ← → West")
    ax.axhline(0, color="gray", ls="--", lw=0.5, alpha=0.5)
    ax.axvline(0, color="gray", ls="--", lw=0.5, alpha=0.5)

    # --- Right panel: Apparent width vs. angle ---
    ax2 = axes[1]
    theta_range = np.linspace(0, 85, 100)
    apparent_width = np.cos(np.radians(theta_range))

    ax2.plot(theta_range, apparent_width, "darkorange", lw=3)
    ax2.fill_between(theta_range, apparent_width, alpha=0.2, color="orange")

    for theta_deg in spot_angles:
        w = np.cos(np.radians(theta_deg))
        ax2.plot(theta_deg, w, "ko", ms=8)
        ax2.annotate(f"{w:.2f}", (theta_deg + 2, w + 0.03), fontsize=9)

    ax2.set_xlabel("Angle from disk center θ (degrees)\n원반 중심에서의 각도 θ (도)")
    ax2.set_ylabel("Apparent width / True width\n겉보기 폭 / 실제 폭")
    ax2.set_title("Foreshortening: $w_{apparent} = w_0 \\cos\\theta$\n원근 단축: $w_{겉보기} = w_0 \\cos\\theta$")
    ax2.set_xlim(0, 90)
    ax2.set_ylim(0, 1.1)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

draw_solar_disk_with_foreshortening()

## Part 2: 3D Visualization — Sunspot on a Rotating Sphere / 3D 시각화 — 회전하는 구 위의 흑점

Let's see the same effect in 3D. A circular spot on the sphere projects
differently depending on its position relative to the observer.

같은 효과를 3D로 봅니다. 구 위의 원형 흑점은 관측자에 대한 위치에 따라
다르게 투영됩니다.

In [ ]:
def plot_3d_sphere_with_spot(rotation_angles):
    """Show a sunspot on a 3D sphere at different rotation angles."""
    fig, axes = plt.subplots(1, len(rotation_angles), figsize=(4 * len(rotation_angles), 4),
                             subplot_kw={"projection": "3d"})

    # Sphere mesh
    u = np.linspace(0, 2 * np.pi, 60)
    v = np.linspace(0, np.pi, 40)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones(np.size(u)), np.cos(v))

    # Sunspot: circle on sphere at latitude=0, longitude=angle
    spot_radius_deg = 8
    spot_phi = np.linspace(0, 2 * np.pi, 40)

    for idx, rot_deg in enumerate(rotation_angles):
        ax = axes[idx]
        rot = np.radians(rot_deg)

        # Draw sphere
        ax.plot_surface(xs, ys, zs, color="#FDB813", alpha=0.4, linewidth=0)

        # Spot center on equator at given longitude
        spot_lat = np.radians(0)
        spot_lon = rot

        # Small circle on sphere surface
        r = np.radians(spot_radius_deg)
        sx = np.cos(spot_phi) * np.sin(r) * np.cos(spot_lon) - \
             np.sin(spot_phi) * np.sin(r) * np.sin(spot_lat) * np.sin(spot_lon) + \
             np.cos(r) * np.cos(spot_lat) * np.cos(spot_lon)
        sy = np.cos(spot_phi) * np.sin(r) * np.sin(spot_lon) + \
             np.sin(spot_phi) * np.sin(r) * np.sin(spot_lat) * np.cos(spot_lon) - \
             np.cos(r) * np.cos(spot_lat) * np.sin(spot_lon)
        # Simplified: place spot points
        sp_lats = np.radians(spot_radius_deg) * np.sin(spot_phi)
        sp_lons = rot + np.radians(spot_radius_deg) * np.cos(spot_phi)
        sx = np.cos(sp_lats) * np.cos(sp_lons)
        sy = np.cos(sp_lats) * np.sin(sp_lons)
        sz = np.sin(sp_lats)

        # Only plot front-facing part
        mask = sx > -0.1
        ax.plot(sx[mask], sy[mask], sz[mask], "k-", lw=3)

        ax.set_xlim([-1.2, 1.2])
        ax.set_ylim([-1.2, 1.2])
        ax.set_zlim([-1.2, 1.2])
        ax.view_init(elev=15, azim=-90)
        ax.set_title(f"Longitude = {rot_deg}°", fontsize=11)
        ax.axis("off")

    fig.suptitle("Sunspot on a Rotating Sphere (observer at x → +∞)\n"
                 "회전하는 구 위의 흑점 (관측자: x → +∞ 방향)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

plot_3d_sphere_with_spot([0, 30, 60, 80])

## Part 3: Apparent Motion on the Solar Disk / 태양 원반 위의 겉보기 운동

Sunspots move fastest at disk center and slow down near the limb.
This is the projection of uniform angular velocity onto a flat plane:

$$x(t) = R\sin(\omega t), \qquad v_{apparent}(t) = R\omega\cos(\omega t)$$

This deceleration near the limb is a natural consequence of uniform rotation
on a sphere — impossible to reproduce with orbiting satellites at constant speed.

흑점은 원반 중심에서 가장 빠르게 이동하고 가장자리에서 느려집니다.
이것은 등각속도 회전을 평면에 투영한 결과입니다.

In [ ]:
def plot_apparent_motion():
    """Plot apparent position and velocity of a sunspot on the solar disk."""
    # Solar rotation period (synodic, as seen from Earth)
    P_synodic = 27.3  # days
    omega = 2 * np.pi / P_synodic  # rad/day

    # Time array: one full transit across visible disk (~13.6 days)
    t_transit = P_synodic / 2
    t = np.linspace(-t_transit / 2, t_transit / 2, 500)

    # Apparent position and velocity
    x = np.sin(omega * t)  # normalized to solar radius
    v = omega * np.cos(omega * t)  # normalized angular velocity

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # Position
    ax1 = axes[0]
    ax1.plot(t, x, "darkorange", lw=3)
    ax1.axhline(0, color="gray", ls="--", lw=0.5)
    ax1.axhline(1, color="red", ls=":", lw=1, label="Limb (가장자리)")
    ax1.axhline(-1, color="red", ls=":", lw=1)
    ax1.set_ylabel("Apparent x-position / $R_{\\odot}$\n겉보기 x-위치")
    ax1.set_title("Sunspot Apparent Motion on Solar Disk\n태양 원반 위의 흑점 겉보기 운동")
    ax1.legend(loc="upper left")
    ax1.grid(True, alpha=0.3)

    # Mark key days
    for day_mark in [-6, -3, 0, 3, 6]:
        x_mark = np.sin(omega * day_mark)
        ax1.plot(day_mark, x_mark, "ko", ms=6)
        ax1.annotate(f"Day {day_mark:+.0f}", (day_mark, x_mark + 0.08),
                     ha="center", fontsize=9)

    # Velocity
    ax2 = axes[1]
    ax2.plot(t, v / omega, "steelblue", lw=3)
    ax2.fill_between(t, v / omega, alpha=0.15, color="steelblue")
    ax2.axhline(0, color="gray", ls="--", lw=0.5)
    ax2.set_xlabel("Time (days from central meridian crossing)\n시간 (중앙 자오선 통과 기준, 일)")
    ax2.set_ylabel("Apparent velocity / $v_{max}$\n겉보기 속도 / $v_{최대}$")
    ax2.set_title("Apparent Velocity: $v = v_0 \\cos(\\omega t)$\n겉보기 속도: 중심에서 최대, 가장자리에서 0")
    ax2.grid(True, alpha=0.3)

    # Annotations
    ax2.annotate("Maximum speed\n(disk center / 원반 중심)",
                 xy=(0, 1), xytext=(3, 0.85),
                 arrowprops=dict(arrowstyle="->", color="black"),
                 fontsize=10, ha="center")
    ax2.annotate("Speed → 0\n(limb / 가장자리)",
                 xy=(t_transit / 2 - 0.5, 0.05), xytext=(t_transit / 2 - 3, 0.4),
                 arrowprops=dict(arrowstyle="->", color="black"),
                 fontsize=10, ha="center")

    plt.tight_layout()
    plt.show()

plot_apparent_motion()

## Part 4: Scheiner vs. Galileo — Two Models Compared / 두 모델 비교

**Scheiner's model**: Sunspots are small satellites orbiting the Sun at some distance.
→ Spots maintain circular shape everywhere. Uniform speed across disk.

**Galileo's model**: Sunspots are on the Sun's surface.
→ Spots are foreshortened near limb. Speed decreases near limb.

Let's simulate both and compare.

**Scheiner 모델**: 흑점은 태양에서 떨어진 궤도를 도는 작은 위성
→ 어디서나 원형 유지, 균일한 속도

**Galileo 모델**: 흑점은 태양 표면 위의 현상
→ 가장자리에서 찌그러짐, 가장자리에서 속도 감소

In [ ]:
def compare_models():
    """Compare Scheiner's satellite model with Galileo's surface model."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    spot_true_radius = 0.07

    # --- Galileo's Surface Model ---
    ax1 = axes[0]
    sun1 = plt.Circle((0, 0), 1.0, color="#FDB813", ec="darkorange", lw=2)
    ax1.add_patch(sun1)

    positions = np.linspace(-80, 80, 9)  # longitude degrees
    for lon_deg in positions:
        lon = np.radians(lon_deg)
        x = np.sin(lon)
        y = 0.15  # slight latitude offset for visibility

        # Surface model: foreshortened
        w = spot_true_radius * np.cos(lon)
        h = spot_true_radius
        ellipse = patches.Ellipse((x, y), 2 * w, 2 * h, color="black", alpha=0.7)
        ax1.add_patch(ellipse)

    ax1.set_xlim(-1.4, 1.4)
    ax1.set_ylim(-1.4, 1.4)
    ax1.set_aspect("equal")
    ax1.set_title("Galileo's Surface Model (CORRECT ✓)\nGalileo 표면 모델 (정확 ✓)\n"
                  "Spots foreshorten near limb / 가장자리에서 찌그러짐",
                  fontsize=12, color="green")
    ax1.set_xlabel("← East    West →")

    # Speed arrows
    arrow_y = -0.3
    for lon_deg in [-60, -30, 0, 30, 60]:
        lon = np.radians(lon_deg)
        x = np.sin(lon)
        speed = np.cos(lon) * 0.15  # proportional arrow length
        ax1.annotate("", xy=(x + speed, arrow_y), xytext=(x, arrow_y),
                     arrowprops=dict(arrowstyle="->", color="blue", lw=2))

    ax1.text(0, -0.5, "Arrow length ∝ apparent speed\n화살표 길이 ∝ 겉보기 속도",
             ha="center", fontsize=10, color="blue")

    # --- Scheiner's Satellite Model ---
    ax2 = axes[1]
    sun2 = plt.Circle((0, 0), 1.0, color="#FDB813", ec="darkorange", lw=2)
    ax2.add_patch(sun2)

    # Satellites orbit at R=1.2 (above surface) — projected onto disk
    R_orbit = 1.15
    for lon_deg in positions:
        lon = np.radians(lon_deg)
        x = R_orbit * np.sin(lon)
        y = 0.15

        if abs(x) < 1.1:  # only show if in front of disk
            # Satellite model: always circular (no foreshortening)
            circle = plt.Circle((x, y), spot_true_radius, color="black", alpha=0.7)
            ax2.add_patch(circle)

    ax2.set_xlim(-1.4, 1.4)
    ax2.set_ylim(-1.4, 1.4)
    ax2.set_aspect("equal")
    ax2.set_title("Scheiner's Satellite Model (WRONG ✗)\nScheiner 위성 모델 (오류 ✗)\n"
                  "Spots stay circular everywhere / 어디서나 원형 유지",
                  fontsize=12, color="red")
    ax2.set_xlabel("← East    West →")

    # Uniform speed arrows
    for lon_deg in [-60, -30, 0, 30, 60]:
        lon = np.radians(lon_deg)
        x = R_orbit * np.sin(lon)
        speed = 0.12  # constant arrow length
        if abs(x) < 1.0:
            ax2.annotate("", xy=(x + speed, arrow_y), xytext=(x, arrow_y),
                         arrowprops=dict(arrowstyle="->", color="blue", lw=2))

    ax2.text(0, -0.5, "Uniform speed (not observed!)\n균일한 속도 (관측과 불일치!)",
             ha="center", fontsize=10, color="red")

    plt.tight_layout()
    plt.show()

compare_models()

## Part 5: Simulating Daily Sunspot Observations / 일일 흑점 관측 시뮬레이션

Simulate what Galileo saw: daily sketches of the solar disk as spots
rotate across from east to west over ~14 days.

Galileo가 본 것을 시뮬레이션합니다: 흑점이 동쪽에서 서쪽으로
~14일에 걸쳐 이동하는 매일의 태양 원반 스케치.

In [ ]:
def simulate_daily_observations():
    """Simulate Galileo-style daily sunspot observation sketches."""
    P_synodic = 27.3  # days
    omega = 2 * np.pi / P_synodic

    # Define several sunspots with (initial_longitude_deg, latitude_deg, radius)
    spots = [
        (-70, 10, 0.08),   # Large spot group
        (-65, 15, 0.04),
        (-55, -5, 0.06),   # Isolated spot
        (-40, 20, 0.05),   # Another group
        (-35, 18, 0.03),
    ]

    days = np.arange(0, 14, 1)  # 14 days of observation
    n_days = len(days)
    ncols = 7
    nrows = (n_days + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.5, nrows * 2.5))
    axes = axes.flatten()

    for i, day in enumerate(days):
        ax = axes[i]

        # Draw solar disk
        sun = plt.Circle((0, 0), 1.0, color="#FDB813", ec="darkorange", lw=1.5)
        ax.add_patch(sun)

        # Draw each sunspot
        for lon0_deg, lat_deg, radius in spots:
            lon_deg = lon0_deg + day * (360.0 / P_synodic)
            lon = np.radians(lon_deg)
            lat = np.radians(lat_deg)

            # Position on disk (orthographic projection)
            x = np.cos(lat) * np.sin(lon)
            y = np.sin(lat)

            # Check if on visible hemisphere
            cos_angle = np.cos(lat) * np.cos(lon)
            if cos_angle <= 0:
                continue  # behind the Sun

            # Foreshortened shape
            w = radius * cos_angle
            h = radius

            ellipse = patches.Ellipse(
                (x, y), 2 * w, 2 * h,
                color="black", alpha=0.85
            )
            ax.add_patch(ellipse)

        ax.set_xlim(-1.3, 1.3)
        ax.set_ylim(-1.3, 1.3)
        ax.set_aspect("equal")
        ax.set_title(f"Day {day + 1}", fontsize=10)
        ax.axis("off")

    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    fig.suptitle("Simulated Daily Sunspot Observations (Galileo-style)\n"
                 "시뮬레이션된 일일 흑점 관측 (Galileo 스타일)",
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

simulate_daily_observations()

## Part 6: Solar Rotation Period Estimation / 태양 자전 주기 추정

Galileo estimated the Sun's rotation period from the transit time of sunspots.
Here we show how to derive the sidereal period from the observed synodic period,
accounting for Earth's orbital motion.

$$\frac{1}{P_{\text{sidereal}}} = \frac{1}{P_{\text{synodic}}} + \frac{1}{P_{\text{Earth}}}$$

Galileo는 흑점의 횡단 시간으로 태양 자전 주기를 추정했습니다.
여기서는 관측된 회합 주기(synodic period)에서 지구 공전을 보정하여
항성 자전 주기(sidereal period)를 구하는 방법을 보여줍니다.

In [ ]:
def estimate_rotation_period():
    """Estimate solar rotation period from sunspot transit observations."""
    # Simulated observation: measure daily x-positions of a sunspot
    P_true_synodic = 27.3  # days (true synodic period)
    omega_true = 2 * np.pi / P_true_synodic

    np.random.seed(42)
    # Observe for 12 days (spot visible for ~14 days total)
    t_obs = np.arange(1, 13)
    # True positions with measurement noise
    x_true = np.sin(omega_true * (t_obs - 7))  # centered at day 7
    x_obs = x_true + np.random.normal(0, 0.02, len(t_obs))
    x_obs = np.clip(x_obs, -0.99, 0.99)

    # Recover angular position: theta = arcsin(x)
    theta_obs = np.arcsin(x_obs)

    # Fit linear model: theta(t) = omega * (t - t0)
    # Using least-squares
    A = np.vstack([t_obs, np.ones(len(t_obs))]).T
    result = np.linalg.lstsq(A, theta_obs, rcond=None)
    omega_fit, offset = result[0]

    P_synodic_est = 2 * np.pi / omega_fit
    P_earth = 365.25  # days
    P_sidereal_est = 1.0 / (1.0 / P_synodic_est + 1.0 / P_earth)

    # True values for comparison
    P_sidereal_true = 25.38  # days (equatorial)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Position fit
    ax1 = axes[0]
    t_fine = np.linspace(0, 14, 200)
    x_fit = np.sin(omega_fit * (t_fine - (-offset / omega_fit)))

    ax1.plot(t_obs, x_obs, "ko", ms=8, label="Observed / 관측값")
    ax1.plot(t_fine, x_fit, "darkorange", lw=2, label="Fit / 적합")
    ax1.plot(t_fine, np.sin(omega_true * (t_fine - 7)), "b--", lw=1,
             alpha=0.5, label="True / 실제")
    ax1.axhline(1, color="red", ls=":", lw=1)
    ax1.axhline(-1, color="red", ls=":", lw=1)
    ax1.set_xlabel("Day of observation / 관측일")
    ax1.set_ylabel("x-position on disk / 원반 위 x-위치")
    ax1.set_title("Fitting Sunspot Transit / 흑점 횡단 적합")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Right: Period comparison
    ax2 = axes[1]
    categories = ["Synodic\n(observed)\n회합 주기\n(관측)",
                  "Sidereal\n(corrected)\n항성 주기\n(보정)",
                  "True sidereal\n(equator)\n실제 항성 주기\n(적도)"]
    values = [P_synodic_est, P_sidereal_est, P_sidereal_true]
    colors = ["#FDB813", "darkorange", "steelblue"]

    bars = ax2.bar(categories, values, color=colors, edgecolor="black", lw=1.2)
    for bar, val in zip(bars, values):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f"{val:.1f} days", ha="center", fontsize=11, fontweight="bold")

    ax2.set_ylabel("Period (days) / 주기 (일)")
    ax2.set_title("Solar Rotation Period Estimation\n태양 자전 주기 추정")
    ax2.set_ylim(0, 32)
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()

    print(f"Estimated synodic period  / 추정 회합 주기:  {P_synodic_est:.1f} days")
    print(f"Estimated sidereal period / 추정 항성 주기:  {P_sidereal_est:.1f} days")
    print(f"True sidereal (equator)   / 실제 (적도):     {P_sidereal_true:.1f} days")
    print(f"Error / 오차: {abs(P_sidereal_est - P_sidereal_true):.1f} days "
          f"({abs(P_sidereal_est - P_sidereal_true) / P_sidereal_true * 100:.1f}%)")

estimate_rotation_period()

## Part 7: Connection to Modern Solar Physics / 현대 태양 물리학과의 연결

Galileo couldn't know it, but the Sun rotates **differentially** — faster at the
equator (~25 days) than at the poles (~35 days). This is because the Sun is not
a rigid body but a fluid (plasma) sphere. Differential rotation is a key driver
of the solar magnetic dynamo.

Galileo는 알 수 없었지만, 태양은 **차등 자전**합니다 — 적도(~25일)가 극(~35일)보다
빠릅니다. 태양이 강체가 아니라 유체(플라즈마) 구이기 때문입니다.
차등 자전은 태양 자기 다이나모의 핵심 동력입니다.

$$\omega(\phi) = A + B\sin^2\phi + C\sin^4\phi$$

where $\phi$ is heliographic latitude, and $A$, $B$, $C$ are empirical constants.

In [ ]:
def plot_differential_rotation():
    """Visualize solar differential rotation — what Galileo could not yet see."""
    # Empirical differential rotation law (Snodgrass & Ulrich 1990)
    # omega(phi) = A + B*sin^2(phi) + C*sin^4(phi)  [deg/day]
    A = 14.713  # deg/day (equatorial rate)
    B = -2.396
    C = -1.787

    lat = np.linspace(0, 85, 200)  # heliographic latitude (degrees)
    lat_rad = np.radians(lat)

    omega = A + B * np.sin(lat_rad)**2 + C * np.sin(lat_rad)**4  # deg/day
    period = 360.0 / omega  # sidereal period in days

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left: Rotation rate vs latitude
    ax1 = axes[0]
    ax1.plot(lat, omega, "darkorange", lw=3)
    ax1.axhline(A, color="red", ls="--", lw=1, alpha=0.5,
                label=f"Equatorial rate: {A:.1f}°/day")
    ax1.fill_between(lat, omega, A, alpha=0.15, color="orange")
    ax1.set_xlabel("Heliographic latitude (°) / 태양면 위도 (°)")
    ax1.set_ylabel("Angular velocity (°/day) / 각속도 (°/일)")
    ax1.set_title("Solar Differential Rotation\n태양 차등 자전")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Galileo's estimate
    ax1.axhspan(12.8, 13.5, alpha=0.2, color="green",
                label="Galileo's estimate (~13°/day)")
    ax1.legend()

    # Right: Rotation period vs latitude
    ax2 = axes[1]
    ax2.plot(lat, period, "steelblue", lw=3)
    ax2.set_xlabel("Heliographic latitude (°) / 태양면 위도 (°)")
    ax2.set_ylabel("Sidereal rotation period (days) / 항성 자전 주기 (일)")
    ax2.set_title("Rotation Period vs. Latitude\n자전 주기 vs. 위도")
    ax2.grid(True, alpha=0.3)

    # Key markers
    for lat_mark, label in [(0, "Equator\n적도"), (30, "30°"), (60, "60°")]:
        p = 360.0 / (A + B * np.sin(np.radians(lat_mark))**2 +
                      C * np.sin(np.radians(lat_mark))**4)
        ax2.plot(lat_mark, p, "ko", ms=8)
        ax2.annotate(f"{label}\n{p:.1f} d", (lat_mark + 2, p),
                     fontsize=10, va="center")

    ax2.set_ylim(24, 38)

    plt.tight_layout()
    plt.show()

    print("Solar differential rotation (Snodgrass & Ulrich 1990):")
    print(f"  Equator (0°):  {360/A:.1f} days  ({A:.2f}°/day)")
    lat_60 = np.radians(60)
    omega_60 = A + B * np.sin(lat_60)**2 + C * np.sin(lat_60)**4
    print(f"  60° latitude:  {360/omega_60:.1f} days  ({omega_60:.2f}°/day)")
    print(f"\nGalileo estimated ~27-28 days (synodic) → consistent with low-latitude spots")

plot_differential_rotation()

## Summary / 요약

| Part | What we showed / 보여준 것 | Key insight / 핵심 통찰 |
|------|--------------------------|------------------------|
| 1 | Foreshortening effect | Spots compress near limb → must be on surface / 가장자리에서 찌그러짐 → 표면 위에 있어야 함 |
| 2 | 3D sphere visualization | Same effect seen from observer's perspective / 관측자 시점에서 본 같은 효과 |
| 3 | Apparent motion & velocity | Speed varies as $\cos\theta$ → uniform rotation on sphere / 속도가 $\cos\theta$로 변화 → 구 위의 등속 회전 |
| 4 | Scheiner vs. Galileo models | Surface model matches observations; satellite model doesn't / 표면 모델은 관측과 일치, 위성 모델은 불일치 |
| 5 | Daily observation simulation | Reproduces Galileo's actual sketches / Galileo의 실제 스케치 재현 |
| 6 | Rotation period estimation | Synodic → sidereal correction; Galileo's estimate was good / 회합→항성 보정; Galileo의 추정은 정확 |
| 7 | Differential rotation | Modern discovery: Sun rotates faster at equator / 현대적 발견: 적도가 더 빨리 자전 |

**Next paper / 다음 논문**: #2 Fraunhofer (1814) — Discovery of solar absorption lines, the foundation of astrophysical spectroscopy.

다음 논문: #2 Fraunhofer (1814) — 태양 흡수선의 발견, 천체 분광학의 기초.